<a href="https://colab.research.google.com/github/Rumeysakeskin/TTS-turkish-emotion/blob/embed/emotion_xtts_base_xtts_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Rumeysakeskin/TTS-turkish-emotion.git
%cd TTS-turkish-emotion
!git checkout embed

In [ ]:
%cd TTS-turkish-emotion
!pip install -e .
!pip install -r requirements.txt
%cd ..
!ls

In [ ]:
# MODEL FILES
!gdown --folder https://drive.google.com/drive/folders/11zHG_bZQqY_zdLqQHeZahjnIumOyxJIF?usp=drive_link

In [ ]:
#REFERENCE DATA
!gdown --folder https://drive.google.com/drive/folders/1QZz9QFJRZf8lpwVY35vjZfWxhQ0SXYUA?usp=sharing

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.36.2

DUYGU TEMELLİ XTTS MODEL INFERENCE

In [ ]:
import os
import subprocess

# ======================================================
# SABİT PATHLER
# ======================================================
INFERENCE_PY = "TTS-turkish-emotion/inference.py"
CONFIG_PATH = "XTTS-Emotion-TR-October-11-2025_06+40AM-44b665b5/config.json"
CHECKPOINT_PATH = "XTTS-Emotion-TR-October-11-2025_06+40AM-44b665b5/best_model.pth"
TOKENIZER_PATH = "XTTS-Emotion-TR-October-11-2025_06+40AM-44b665b5/vocab.json"

REFERENCE_AUDIO_DIR = "/references"
OUTPUT_DIR = "xtts-output-19-12-2025"

EMOTION = "happy"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================================================
# REFERANS SESLER VE METİNLER
# ======================================================
references = {
    "audio1.wav": "Ancak yaratığın adı basit kalırken itibarı kısa sürede oldukça karmaşık hale geldi.",
    "audio2.wav": "Bir limonatacıda beş dakika oturmaya razı oldu ve hikayesine orada devam etti.",
    "audio3.wav": "Aslında bunu yapmaktaki amaç kölelik ve onun etrafında şekillenen dilin günümüzde hala bizim hayatımızın bir parçası olduğunu göstermektedir.",
    "audio4.wav": "Bankın tepesinde ellerin neredeyse kavuşuyor olduğu hariç ikisi arasında bir bariyer gibi kullanılmış olması detayı çok hoş.",
    "audio5.wav": "Reel faiz oranı ve reel gayri safi hasılı açısından dengede dedik. Şimdi bir düşünelim. Merkez bankası fazladan para basmaya kalkarsa ne olur?",
    "audio6.wav": "Hızlı konuşma problemi nasıl çözülür? Bununla ilgili birkaç tane tavsiye verdim. Çok hızlı konuşan insanlar genelde çok hızlı düşünen insanlardır.",
    "audio7.wav": "Son olarak taa şuralara inerseniz fiyatlarımız oldukça aşağıda olacak. Fiyatlar oldukça aşağıda. Bu durumda bir dolarlık bir değişim çok büyük yüzdesel fiyat değişimi değil mi?",
    "audio8.wav": "Yine fiziki altınlarımızda altın parçalarıyla herhangi bir işlem yapmadık. Hemen akla şu soru gelebilir. Bu işlemler nereye kadar devam edebilir?",
    "audio9.wav": "Tarih boyunca sanat, büyük soruları cevaplamaya çalışmış ve burada gördüğümüz sanatsal çalışma da bu önemli soruları sormaya devam ediyor.",
    "audio10.wav": "Veya film rulosu kabınız yoksa, kendi yaptığınız siyah kabınızı sabunlu suya batırıp beyaz kağıdın üstüne koyabilirsiniz.",
    "audio11.wav": "Kırmızı ampulden çıkan ışık bu şekilde tahtaya ilerliyor, ama kalem onu bu şekilde engelliyor.",
    "audio12.wav": "Yine algılanabilir hale getirmek için biz bu resmin neresindeyiz göstereyim.",
    "audio13.wav": "Rutin hayattan alınmış, yükselebilen ve alçalabilen, çelişkiden uzak, göz kamaştırıcı bir tecrübe.",
    "audio14.wav": "Dolayısıyla da bu dönemden günümüze ulaşabilen bronz heykel sayısı son derece az, burada gördüğümüz de o nadir örneklerden birisi. Şu an görmekte olduğumuz, Milattan Önce yüz yıllarından kalmış bir heykel. 'Dinlenen Boksör' heykeline bakıyoruz.",
    "audio15.wav": "Bize insan doğası hakkında da çok şey söylüyor, bu sürekli ve daimi güzellik arayışına dair. Zaman geçtikçe parlaklığı azalabilir ama bu arayış hiç solmayacak.",
    "audio16.wav": "Bu insanların, yani Atinalıların vahşi doğayı kontrol edecek güce sahip olduğu simgeleniyor bu frizede. Atlar vahşi doğayı temsil ediyor.",
    "audio17.wav": "Hücre dışındaki bu pozitif yükler kendileriyle aynı yükten olan diğer iyonlardan uzaklaşmak ve daha negatif yüklü olan tarafa doğru hareket etmek isteyecekler.",
    "audio18.wav": "Eğer yüz derecede bir gram su buharı varsa ve bunu yoğunlaştırmak istersem, sistemden bu kadar enerji almam gerekir.",
    "audio19.wav": "Kültürün bu iki cephesinin birbirine tamamen zıt olması dolayısıyla, yeni bir teknolojinin kabul edilmesi de zor olur.",
    "audio20.wav": "Önceki videoda galiba son olarak Güneş'in Dünya'ya göre ne kadar büyük olduğunu ve Dünya'nın Güneş'ten ne kadar uzak olduğunu görmüştük.",
}

# ======================================================
# INFERENCE
# ======================================================
missing_files = []

for audio_name, text in references.items():
    speaker_wav = os.path.join(REFERENCE_AUDIO_DIR, audio_name)

    if not os.path.isfile(speaker_wav):
        print(f"Referans ses bulunamadı: {speaker_wav}")
        missing_files.append(audio_name)
        continue

    output_wav = os.path.join(OUTPUT_DIR, audio_name)

    cmd = [
        "python", INFERENCE_PY,
        "--config", CONFIG_PATH,
        "--checkpoint", CHECKPOINT_PATH,
        "--tokenizer", TOKENIZER_PATH,
        "--speaker-wav", speaker_wav,
        "--text", text,
        "--emotion", EMOTION,
        "--output", output_wav
    ]

    print(f"▶Üretiliyor: {audio_name}")
    subprocess.run(cmd, check=True)

# ======================================================
# RAPOR
# ======================================================
print("\n==================== RAPOR ====================")

if missing_files:
    print("Üretilmeyen dosyalar (bulunamadı):")
    for f in missing_files:
        print(" -", f)
else:
    print("Tüm referans sesler başarıyla işlendi.")

print("================================================")


ORIGINAL XTTS MODEL INFERENCE

In [ ]:
!git clone https://github.com/coqui-ai/TTS
%cd TTS
!pip install -e .

In [ ]:
# =====================================================
# 0. IMPORTS + TORCH SAFE GLOBALS (ZORUNLU)
# =====================================================
import os
import time
import torch
import torchaudio

from torch.serialization import add_safe_globals
from huggingface_hub import snapshot_download

from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts, XttsAudioConfig
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.models.xtts import XttsArgs

# PyTorch 2.6 SAFE allowlist
add_safe_globals([
    XttsConfig,
    XttsAudioConfig,
    BaseDatasetConfig,
    XttsArgs,
])



# =====================================================
# 1. PATHLER
# =====================================================
REFERENCE_AUDIO_DIR = "references"
OUTPUT_DIR = "base-output-19-12-2025"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================
# 2. REFERENCES
# =====================================================
references = {
    "audio1.wav": "Ancak yaratığın adı basit kalırken itibarı kısa sürede oldukça karmaşık hale geldi.",
    "audio2.wav": "Bir limonatacıda beş dakika oturmaya razı oldu ve hikayesine orada devam etti.",
    "audio3.wav": "Aslında bunu yapmaktaki amaç kölelik ve onun etrafında şekillenen dilin günümüzde hala bizim hayatımızın bir parçası olduğunu göstermektedir.",
    "audio4.wav": "Bankın tepesinde ellerin neredeyse kavuşuyor olduğu hariç ikisi arasında bir bariyer gibi kullanılmış olması detayı çok hoş.",
    "audio5.wav": "Reel faiz oranı ve reel gayri safi hasılı açısından dengede dedik. Şimdi bir düşünelim. Merkez bankası fazladan para basmaya kalkarsa ne olur?",
    "audio6.wav": "Hızlı konuşma problemi nasıl çözülür? Bununla ilgili birkaç tane tavsiye verdim. Çok hızlı konuşan insanlar genelde çok hızlı düşünen insanlardır.",
    "audio7.wav": "Son olarak taa şuralara inerseniz fiyatlarımız oldukça aşağıda olacak. Fiyatlar oldukça aşağıda. Bu durumda bir dolarlık bir değişim çok büyük yüzdesel fiyat değişimi değil mi?",
    "audio8.wav": "Yine fiziki altınlarımızda altın parçalarıyla herhangi bir işlem yapmadık. Hemen akla şu soru gelebilir. Bu işlemler nereye kadar devam edebilir?",
    "audio9.wav": "Tarih boyunca sanat, büyük soruları cevaplamaya çalışmış ve burada gördüğümüz sanatsal çalışma da bu önemli soruları sormaya devam ediyor.",
    "audio10.wav": "Veya film rulosu kabınız yoksa, kendi yaptığınız siyah kabınızı sabunlu suya batırıp beyaz kağıdın üstüne koyabilirsiniz.",
    "audio11.wav": "Kırmızı ampulden çıkan ışık bu şekilde tahtaya ilerliyor, ama kalem onu bu şekilde engelliyor.",
    "audio12.wav": "Yine algılanabilir hale getirmek için biz bu resmin neresindeyiz göstereyim.",
    "audio13.wav": "Rutin hayattan alınmış, yükselebilen ve alçalabilen, çelişkiden uzak, göz kamaştırıcı bir tecrübe.",
    "audio14.wav": "Dolayısıyla da bu dönemden günümüze ulaşabilen bronz heykel sayısı son derece az, burada gördüğümüz de o nadir örneklerden birisi.",
    "audio15.wav": "Bize insan doğası hakkında da çok şey söylüyor, bu sürekli ve daimi güzellik arayışına dair.",
    "audio16.wav": "Bu insanların, yani Atinalıların vahşi doğayı kontrol edecek güce sahip olduğu simgeleniyor bu frizede.",
    "audio17.wav": "Hücre dışındaki bu pozitif yükler kendileriyle aynı yükten olan diğer iyonlardan uzaklaşmak isteyecekler.",
    "audio18.wav": "Eğer yüz derecede bir gram su buharı varsa ve bunu yoğunlaştırmak istersem, sistemden bu kadar enerji almam gerekir.",
    "audio19.wav": "Kültürün bu iki cephesinin birbirine tamamen zıt olması dolayısıyla, yeni bir teknolojinin kabul edilmesi de zor olur.",
    "audio20.wav": "Önceki videoda galiba son olarak Güneş'in Dünya'ya göre ne kadar büyük olduğunu görmüştük.",
}

# =====================================================
# 3. DOWNLOAD XTTS-v2
# =====================================================
print("Downloading XTTS-v2...")
checkpoint_path = snapshot_download(
    repo_id="coqui/XTTS-v2",
    local_dir_use_symlinks=False
)

config_path = os.path.join(checkpoint_path, "config.json")

# =====================================================
# 4. LOAD CONFIG + MODEL
# =====================================================
print("Loading config...")
config = XttsConfig()
config.load_json(config_path)

print("Initializing model...")
model = Xtts.init_from_config(config)

model.load_checkpoint(
    config,
    checkpoint_dir=checkpoint_path,
    use_deepspeed=False,
    eval=True
)

model.cuda()
model.eval()

# =====================================================
# 5. BATCH INFERENCE
# =====================================================
for audio_name, text in references.items():
    ref_path = os.path.join(REFERENCE_AUDIO_DIR, audio_name)

    if not os.path.isfile(ref_path):
        print(f"Referans ses yok: {ref_path}")
        continue

    print(f"Processing: {audio_name}")

    # ---- Speaker conditioning
    gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
        audio_path=[ref_path]
    )

    # ---- Streaming inference
    t0 = time.time()
    chunks = model.inference_stream(
        text=text,
        language="tr",
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
    )

    wav_chunks = []

    for i, chunk in enumerate(chunks):
        if i == 0:
            print(f"Time to first chunk: {time.time() - t0:.2f}s")
        wav_chunks.append(chunk)

    wav = torch.cat(wav_chunks, dim=0)

    out_path = os.path.join(OUTPUT_DIR, audio_name)

    torchaudio.save(
        out_path,
        wav.unsqueeze(0).cpu(),
        24000
    )

    print(f"Saved: {out_path}")

print("\n🎉 Tüm XTTS-v2 inference işlemleri tamamlandı.")
